In [0]:
# imports and read bronze
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

df = spark.table("ipl_catalog.bronze.deliveries")
print(f"Bronze matches row count: {df.count()}")

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("is_boundary", 
        F.when(F.col("batsman_runs").isin(4, 6), True).otherwise(False)) \
       .withColumn("is_six", 
        F.when(F.col("batsman_runs") == 6, True).otherwise(False)) \
       .withColumn("is_four", 
        F.when(F.col("batsman_runs") == 4, True).otherwise(False)) \
       .withColumn("is_dot_ball", 
        F.when(F.col("total_runs") == 0, True).otherwise(False)) \
       .withColumn("is_wicket_flag", 
        F.when(F.col("is_wicket") == 1, True).otherwise(False)) \
       .withColumn("is_death_over", 
        F.when(F.col("over") >= 16, True).otherwise(False)) \
       .withColumn("is_powerplay", 
        F.when(F.col("over") <= 6, True).otherwise(False))

df.show(5)

In [0]:
#handling extras properly
df = df.withColumn(
    "is_legal_delivery",
    F.when(F.col("extras_type").isin("wides", "noballs"), False)
     .otherwise(True)
)

df = df.withColumn(
    "extras_type",
    F.when(F.col("extras_type").isNull(), "none")
     .otherwise(F.col("extras_type"))
)

df.select("extras_type", "is_legal_delivery", "extra_runs").show(10)

In [0]:
# join with silver.matches to bring in season context
matches_lookup = spark.table("ipl_catalog.silver.matches") \
    .select("id", "season_year", "venue", "season")

df = df.join(
    matches_lookup,
    df.match_id == matches_lookup.id,
    "left"
).drop("id")

df.select("match_id", "batting_team", "season_year", "venue").show(5)

In [0]:
orphan_count = df.filter(F.col("season_year").isNull()).count()
print(f"Deliveries with no matching match record: {orphan_count}")

if orphan_count > 0:
    print("⚠ Warning: orphan records found - investigate before proceeding")
else:
    print("✓ Referential integrity check passed")

In [0]:
# write to silver
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("season_year") \
    .saveAsTable("ipl_catalog.silver.deliveries")

spark.sql("""
    ALTER TABLE ipl_catalog.silver.deliveries
    SET TBLPROPERTIES (
        'comment' = 'Cleaned ball-by-ball data with derived flags, partitioned by season',
        'layer' = 'silver'
    )
""")

print(f"✓ silver.deliveries written: {df.count()} rows")

In [0]:
spark.table("ipl_catalog.silver.deliveries") \
    .select("batting_team", "is_boundary", "is_dot_ball", 
            "is_death_over", "season_year") \
    .show(5, truncate=False)